In [1]:
import os
def _set_env_from_file(var: str, file_path: str = "openai_key.txt"):
    """
    Reads an API key from a specified file and sets it as an environment variable.
    """
    if not os.environ.get(var):
        try:
            # The 'with open' statement ensures the file is closed automatically
            with open(file_path, 'r') as f:
                # Read the first line and strip any leading/trailing whitespace
                key = f.readline().strip()

            if key:
                os.environ[var] = key
                print(f"Successfully loaded {var} from {file_path}")
            else:
                print(f"Warning: {file_path} is empty.")

        except FileNotFoundError:
            print(f"Error: Key file not found at {file_path}. Please create the file.")

# --- Execution ---
# Set the environment variable OPENAI_API_KEY from the file
_set_env_from_file('OPENAI_API_KEY',file_path="../../keys/openai.txt")

Successfully loaded OPENAI_API_KEY from ../../keys/openai.txt


In [2]:
MODEL="gpt-3.5-turbo-0125"

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv

In [4]:
load_dotenv()


False

In [5]:
llm = ChatOpenAI(model_name=MODEL,)

In [6]:
import pandas as pd

file_path=("./data/cus.csv")
data=pd.read_csv(file_path,)

In [7]:
data.head()

,Index,Customer Id,First Name,Last Name,Company,City,Country,Phone 1,Phone 2,Email,Subscription Date,Website
0,1,DD37Cf93aecA6Dc,Sheryl,Baxter,Rasmussen Group,East Leonard,Chile,229.077.5154,397.884.0519x718,zunigavanessa@smith.info,2020-08-24,http://www.stephenson.com/
1,2,1Ef7b82A4CAAD10,Preston,Lozano,Vega-Gentry,East Jimmychester,Djibouti,5153435776,686-620-1820x944,vmata@colon.com,2021-04-23,http://www.hobbs.com/
2,3,6F94879bDAfE5a6,Roy,Berry,Murillo-Perry,Isabelborough,Antigua and Barbuda,+1-539-402-0259,(496)978-3969x58947,beckycarr@hogan.com,2020-03-25,http://www.lawrence.com/
3,4,5Cef8BFA16c5e3c,Linda,Olsen,"Dominguez, Mcmillan and Donovan",Bensonview,Dominican Republic,001-808-617-6467x12895,+1-813-324-8756,stanleyblackwell@benson.org,2020-06-02,http://www.good-lyons.com/
4,5,053d585Ab6b3159,Joanna,Bender,"Martin, Lang and Andrade",West Priscilla,Slovakia (Slovak Republic),001-234-203-0635x76146,001-199-446-3860x3486,colinalvarado@miles.net,2021-04-17,https://goodwin-ingram.com/


In [8]:
loader = CSVLoader(file_path=file_path, 
            )
docs = loader.load_and_split()

In [9]:
#Initiate Faiss vector store and openai embedding

import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
index = faiss.IndexFlatL2(len(OpenAIEmbeddings().embed_query(" ")))

vector_store = FAISS(
    embedding_function=OpenAIEmbeddings(),
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [10]:
vector_store.add_documents(documents=docs)

['17fa1c9b-87ce-4a5e-a494-f4a32b3f1056',
 '95458ebb-4969-445d-8df4-2bd63261b6b6',
 'f2c9de2b-d3f2-4ca1-be1b-daeedd8c9b4a',
 '9fecb279-f1ac-42c8-b593-879fedd886d8',
 'ba1737a7-484d-4f18-9452-ea1d55f17e63',
 'd541ccbf-caeb-42c7-a7d0-6eda6451f9e2',
 'f96b9937-1c5f-4e84-b4b0-29931917dc3e',
 '6d610743-b49d-4de4-b4bf-81ff49c49fda',
 'aa137a9c-ae01-48d8-8599-b7826fe178dd',
 '0bd3dde5-b370-4ec2-8998-5376419ebf55',
 '002560b2-3df4-412d-87f5-993bb5f6052e',
 'e070f7f3-68c6-4851-abf9-7e1be0b66935',
 '9fccdc63-6147-4aa7-b2b8-587729d94818',
 '4c648ebb-c7b8-4efc-b033-663086db9b01',
 '4e178f9c-2968-4cd6-9179-dd73f033c7d2',
 'fc29ba0d-a2e6-4e70-982f-1357b554425e',
 '7967939f-2066-4709-8600-9c4a3903003e',
 '53f12d2b-1996-4ff0-af12-63abc8433784',
 '5df35a3a-ef0a-4c78-a560-b6f5328c1afe',
 '8fbfcb54-8283-4d88-8ef5-4995df5ed39b',
 'dd2c4058-6be5-4274-8ea2-93fbdb3bd3af',
 '849e29a6-d7ed-41c0-837b-b0d186333328',
 'ae8e2ac0-28c1-4346-a80c-21a4cfafc9f1',
 '9af43e9d-d4d1-45b8-b650-020b12a95100',
 '031e6567-711b-

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


In [12]:
retriever = vector_store.as_retriever()

#system prompt

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [13]:
#create the question-answer chain

question_answer_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [14]:
answer = rag_chain.invoke({
    "input" : "Which company does Brady  work for?"
})
answer['answer']


'Brady Mccann works for Hobbs, Garrett and Sanford. Brady Cohen works for Osborne-Erickson.'

In [24]:
answer

{'input': 'Which company does sheryl Baxter work for?',
 'context': [Document(id='69530ee5-7f34-4dd4-b737-0975eb33e99b', metadata={'source': './data/cus.csv', 'row': 0}, page_content='Index: 1\nCustomer Id: DD37Cf93aecA6Dc\nFirst Name: Sheryl\nLast Name: Baxter\nCompany: Rasmussen Group\nCity: East Leonard\nCountry: Chile\nPhone 1: 229.077.5154\nPhone 2: 397.884.0519x718\nEmail: zunigavanessa@smith.info\nSubscription Date: 2020-08-24\nWebsite: http://www.stephenson.com/'),
  Document(id='35540ccd-e961-4d7f-be7f-b56bc07ba469', metadata={'source': './data/cus.csv', 'row': 8}, page_content='Index: 9\nCustomer Id: C2dE4dEEc489ae0\nFirst Name: Sheryl\nLast Name: Meyers\nCompany: Browning-Simon\nCity: Robersonstad\nCountry: Cyprus\nPhone 1: 854-138-4911x5772\nPhone 2: +1-448-910-2276x729\nEmail: mariokhan@ryan-pope.org\nSubscription Date: 2020-01-13\nWebsite: https://www.bullock.net/'),
  Document(id='b5901c38-4092-403e-ac4b-523a4028486a', metadata={'source': './data/cus.csv', 'row': 61}, pa

In [29]:
answer['answer']

'Sheryl Baxter works for Rasmussen Group in Chile.'